In [18]:
import numpy as np

data = np.load("../data/processed/prepared.npz")
X_train, X_test = data["X_train"], data["X_test"]
y_train, y_test = data["y_train"], data["y_test"]
print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Тип:", X_train.dtype)   # ожидаем float64, а не object

Train: (175341, 194) | Test: (82332, 194)
Тип: float64


In [19]:
import numpy as np
import pandas as pd
import joblib

data = np.load("../data/processed/prepared.npz")
X_train, X_test = data["X_train"], data["X_test"]
y_train, y_test = data["y_train"], data["y_test"]
print("Train:", X_train.shape, "| Test:", X_test.shape)

Train: (175341, 194) | Test: (82332, 194)


In [20]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
print("Обучение завершено")

Обучение завершено


In [21]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report)

y_pred_rf = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]

print(f"Accuracy:  {accuracy_score(y_test, y_pred_rf):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_rf):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_rf):.4f}")
print(f"F1-score:  {f1_score(y_test, y_pred_rf):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_proba_rf):.4f}")

print("\nМатрица ошибок:")
print(confusion_matrix(y_test, y_pred_rf))


Accuracy:  0.8713
Precision: 0.8177
Recall:    0.9860
F1-score:  0.8940
ROC-AUC:   0.9793

Матрица ошибок:
[[27035  9965]
 [  635 44697]]


In [22]:
# Загружаем baseline (LogReg), чтобы сравнить на ОДНОМ тесте
logreg = joblib.load("../models/baseline_logreg.joblib")
y_pred_lr = logreg.predict(X_test)
y_proba_lr = logreg.predict_proba(X_test)[:, 1]

def get_metrics(y_pred, y_proba):
    return {
        "accuracy": round(accuracy_score(y_test, y_pred), 4),
        "precision": round(precision_score(y_test, y_pred), 4),
        "recall": round(recall_score(y_test, y_pred), 4),
        "f1": round(f1_score(y_test, y_pred), 4),
        "roc_auc": round(roc_auc_score(y_test, y_proba), 4),
    }

comparison = pd.DataFrame({
    "LogReg (baseline)": get_metrics(y_pred_lr, y_proba_lr),
    "RandomForest": get_metrics(y_pred_rf, y_proba_rf),
})
print(comparison)

comparison.to_csv("../reports/model_comparison.csv", encoding="utf-8-sig")
joblib.dump(rf, "../models/random_forest.joblib")
print("\nСохранено: reports/model_comparison.csv, models/random_forest.joblib")

FileNotFoundError: [Errno 2] No such file or directory: '../models/baseline_logreg.joblib'